In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv('train.csv')
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [4]:
df.shape

(891, 12)

In [4]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [5]:
df.head(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C


In [6]:
df.shape

(891, 8)

In [7]:
x_train, x_test , y_train , y_test = train_test_split(df.drop(columns=['Survived']),
                                                       df['Survived'],test_size=0.2,random_state=42)

In [8]:
x_train.shape

(712, 7)

In [9]:
y_train.shape

(712,)

In [10]:
x_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S
733,2,male,23.0,0,0,13.0,S


In [11]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [ ]:
#ColumnTransformer for imputation
trf1 = ColumnTransformer([
    ('Impute_age',SimpleImputer(),[2]),
    ('Impute_embark',SimpleImputer(strategy='most_frequent'),[6])
],remainder='passthrough')

In [17]:
#columntransformer for onehotencoding for sex and embarked feature 
trf2 = ColumnTransformer([
    ('ohe_sex_embark',OneHotEncoder(sparse_output=False , handle_unknown='ignore'),[1,6])
],remainder='passthrough')

In [18]:
#scaling of all columns
trf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
])

In [ ]:
#Feature Selection
trf4 = SelectKBest(score_func=chi2, k=8)

In [20]:
trf5 = DecisionTreeClassifier()

In [21]:
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5) 
])

In [ ]:
#alternate syntax
#pipe = Pipeline(trf1,trf2,trf3,trf4,tr5)

In [22]:
pipe.fit(x_train,y_train)

,steps,"[('trf1', ...), ('trf2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Impute_age', ...), ('Impute_embark', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [27]:
y_pred = pipe.predict(x_test)

In [28]:
from sklearn.metrics import accuracy_score

In [29]:
accuracy_score(y_test,y_pred)

0.6256983240223464

Cross validation Using pipelines

In [36]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe ,x_train,y_train , cv=5 , scoring='accuracy')

array([0.6013986 , 0.62237762, 0.68309859, 0.65492958, 0.63380282])

GridSearch using pipelines

In [39]:
params = {
    'trf5__max_depth':[1,2,3,4,5,None]
}

In [40]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe,params,cv=5 , scoring='accuracy')
grid.fit(x_train , y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'trf5__max_depth': [1, 2, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Impute_age', ...), ('Impute_embark', ...)]"


Exporting Pipeline

In [41]:
import pickle
pickle.dump(pipe,open('Pipe.pkl','wb'))